In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from momentfm import MOMENTPipeline
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.knn import KNN
from dtaianomaly.data import UCRLoader
from dtaianomaly.evaluation import VolumeUnderPR
from dtaianomaly.visualization import plot_demarcated_anomalies

## Data

In [ ]:
# The proof-of-concept case demonstrated in the paper, 
# which corresponds to the last 4184 data points of the file 474_UCR_id_172_Environment_tr_2046_1st_6520.csv in the TSB-AD-U dataset.
# Use the commented-out cell below if you want to load the entire file instead of just the last 4184 data points.
dataset = UCRLoader('007_UCR_Anomaly_DISTORTEDCIMIS44AirTemperature3_4000_6520_6544.txt').load()

X_test, y_test = dataset.X_test, dataset.y_test
plot_demarcated_anomalies(X_test, y_test, figsize=(20, 2));

In [ ]:
# file_name = '474_UCR_id_172_Environment_tr_2046_1st_6520.csv'
# df = pd.read_csv(f'../../data/TSB-AD-U/{file_name}').dropna()

# X_test = df['Data'].to_numpy()
# y_test = df['Label'].to_numpy()

# plot_demarcated_anomalies(X_test, y_test, figsize=(20, 2));

In [ ]:
df = pd.read_csv('../uni_mergedTable_VUS-PR.csv', index_col='file')
irrelevant_columns = ['ts_len', 'anomaly_len', 'num_anomaly', 'avg_anomaly_len', 'anomaly_ratio', 'point_anomaly', 'seq_anomaly']
results = df.loc['474_UCR_id_172_Environment_tr_2046_1st_6520.csv'].drop(irrelevant_columns).sort_values(ascending=False)

model_of_interest = 'MOMENT (ZS)'
ax = results.plot(
    kind='bar', 
    figsize=(20, 2),
    title=f'Rank {model_of_interest}: {results.rank(ascending=False)[model_of_interest]:1.0f}',
    ylim=[0, 1.1]
)
for p, label in zip(ax.patches, results.index):
    ax.annotate(f'{p.get_height()*100:2.3f}', (p.get_x(), p.get_height() + 0.01))
    if label == model_of_interest:
        p.set_color("red")

## Load MOMENT

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

moment = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-base", 
    model_kwargs={'task_name': 'embedding'}, # We are loading the model in `embedding` mode to learn representations
)
EMBEDDING_DIMENSION_MOMENT = 768  # 1024 for 'large'
moment.init()
moment = moment.to(device).float()

## Create embeddings

In [ ]:
def _create_batches_and_mask(X: np.ndarray, window_size, batch_size) -> list[torch.tensor]:
    X = X.squeeze()

    nb_windows = X.shape[0] - window_size + 1
    nb_complete_batches = nb_windows // batch_size
    nb_remaining_windows = nb_windows - nb_complete_batches * batch_size

    batches = []
    for i in range(nb_complete_batches):
        batches.append(np.array(
                [
                    X[(i * batch_size) + t : (i * batch_size) + t + window_size].T
                    for t in range(batch_size)
                ]
            )
        )

    if nb_remaining_windows > 0:
        batches.append(np.array(
                [
                    X[(nb_complete_batches * batch_size) + t : (nb_complete_batches * batch_size) + t + window_size].T
                    for t in range(nb_remaining_windows)
                ]
            )
        )
        
    batches = [torch.tensor(batch).unsqueeze(1).to(device).float() for batch in batches]

    return batches

In [ ]:
def compute_embedding(X: np.array, window_size: int):
    batches = _create_batches_and_mask(X, window_size, 16)
    moment.eval()
    embeddings = np.full(shape=(X_test.shape[0], EMBEDDING_DIMENSION_MOMENT), fill_value=np.nan)
    with torch.no_grad():
        for i, batch in enumerate(batches):
            outputs = moment(x_enc=batch)
            start = i * len(batches[0]) + batch[0].shape[1] // 2 
            end = start + len(batch)
            embeddings[start:end,:] = outputs.embeddings.cpu()
    embeddings[:batches[0][0].shape[1] // 2] = embeddings[batches[0][0].shape[1] // 2]
    embeddings[-batches[0][0].shape[1] // 2:] = embeddings[-batches[0][0].shape[1] // 2 - 1]
    
    return embeddings

In [ ]:
embedding = compute_embedding(X_test, 64)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 6), sharex='all')
plot_demarcated_anomalies(X_test, y_test, ax=ax1)
ax2.imshow(embedding.T, aspect='auto', cmap='gist_gray_r')
fig.tight_layout()

# Detect anomalies

In [ ]:
random_state = 0
detectors = [ECOD(), IForest(random_state=random_state), LOF(), COPOD(), KNN()]
fig, axs = plt.subplots(len(detectors) + 1, figsize=(20, 10), sharex='all')
plot_demarcated_anomalies(X_test, y_test, ax=axs[0])
for i, detector in enumerate(detectors):
    detector.fit(embedding)
    y_pred = detector.decision_function(embedding)
    axs[i+1].plot(y_pred)
    vus_pr = VolumeUnderPR().compute(y_test, y_pred)
    axs[i+1].set_title(f'{detector.__class__.__name__} (VUS-PR={vus_pr:.5f})')
fig.tight_layout()

## Small tuning

In [ ]:
n_neighbors = [1, 3, 5, 10, 20, 50]
fig, axs = plt.subplots(len(n_neighbors) + 1, figsize=(20, 10), sharex='all')
plot_demarcated_anomalies(X_test, y_test, ax=axs[0])
for i, k in enumerate(n_neighbors):
    detector = KNN(n_neighbors=k)
    detector.fit(embedding)
    y_pred = detector.decision_function(embedding)
    axs[i+1].plot(y_pred)
    vus_pr = VolumeUnderPR().compute(y_test, y_pred)
    axs[i+1].set_title(f'n_neighbors={k} (VUS-PR={vus_pr:.5f})')
fig.tight_layout()

## Show the results

In [ ]:
detector = KNN(n_neighbors=20)
detector.fit(embedding)
y_pred = detector.decision_function(embedding)
vus_pr = VolumeUnderPR().compute(y_test, y_pred)

results['Embedding-based'] = vus_pr
results = results.sort_values(ascending=False)
model_of_interest = 'Embedding-based'
ax = results.plot(
    kind='bar', 
    figsize=(20, 2),
    title=f'Rank {model_of_interest}: {results.rank(ascending=False)[model_of_interest]:1.0f}',
    ylim=[0, 1.1]
)
for p, label in zip(ax.patches, results.index):
    ax.annotate(f'{p.get_height()*100:2.3f}', (p.get_x(), p.get_height() + 0.01))
    if label == model_of_interest:
        p.set_color("green")
    if label == 'MOMENT (ZS)':
        p.set_color("red")

## Store the results

In [ ]:
np.save("embedding_based", y_pred)

In [ ]:
with open('metrics.json', 'w') as f:
    json.dump(
        {
            'MOMENT (ZS)': {
                'VUS-PR': results['MOMENT (ZS)'],
                'rank': int(results.rank(ascending=False)['MOMENT (ZS)']) - 1,  # -1 to account for Embedding-based 
            },
            'Embedding-based': {
                'VUS-PR': results['Embedding-based'],
                'rank': int(results.rank(ascending=False)['Embedding-based']),
            }
        }, 
        f
    )